# Packages

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split,cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("../data/processed/RDC_Inventory_Core_Metrics_Zip_History.csv") 

# url = "https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_Zip_History.csv"
# data =  pd.read_csv(url)

# XGB

### Define target + features

In [39]:
## Takes about ~32 seconds on GPU

# ---------------------------
# 1. Define target + features
# ---------------------------
target_col = "median_listing_price"  # change if needed

df["price_lag_1"] = df["median_listing_price"].shift(1)
df["price_roll_3"] = df["median_listing_price"].shift(1).rolling(3).mean()

# Drop non-numeric columns (RF can’t handle strings directly)
# Drop rows where target is NaN
df_clean = df.dropna(subset=["median_listing_price"])
df_clean = df_clean.drop(columns=["average_listing_price","average_listing_price_mm","average_listing_price_yy"
                                  ,"median_listing_price_per_square_foot", "median_listing_price_per_square_foot_mm","median_listing_price_per_square_foot_yy"])

X = df_clean.drop(columns=["median_listing_price"])
y = df_clean["median_listing_price"]

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

### Train/Test Split

In [40]:
# ---------------------------
# 2. Train/Test Split
# ---------------------------

####Learn this more
f = df.sort_values("month_date_yyyymm")

split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

### Pipeline

In [ ]:
# ---------------------------
# 3. Pipeline
# ---------------------------
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    
    # Remove low-variance features
    ("variance", VarianceThreshold(threshold=0.01)),
    
    # Optional for RF/XGB?(confirm), but keeps things consistent
    ("scaler", StandardScaler()),
    
    ("model", XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    device="cuda",  # GPU enabled
    random_state=42
))
])

# ---------------------------
# 4. Train
# ---------------------------
pipeline.fit(X_train, y_train)

# ---------------------------
# 5. Predict + Evaluate
# ---------------------------
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

# Get mask of selected features
    #variance_mask = pipeline.named_steps["variance"].get_support()

#selected_features = X.columns[variance_mask]
    #print("Selected Features:")
    #print(selected_features)

c:\Users\mduro\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:729: UserWarning: [16:58:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


RMSE: 545168.20
R²: 0.3943
Selected Features:
Index(['month_date_yyyymm', 'postal_code', 'median_listing_price_mm',
       'median_listing_price_yy', 'active_listing_count',
       'active_listing_count_mm', 'active_listing_count_yy',
       'median_days_on_market', 'median_days_on_market_mm',
       'median_days_on_market_yy', 'new_listing_count', 'new_listing_count_mm',
       'new_listing_count_yy', 'price_increased_count',
       'price_increased_count_mm', 'price_increased_count_yy',
       'price_reduced_count', 'price_reduced_count_mm',
       'price_reduced_count_yy', 'price_reduced_share',
       'price_reduced_share_mm', 'price_reduced_share_yy',
       'pending_listing_count', 'pending_listing_count_mm',
       'pending_listing_count_yy', 'median_square_feet',
       'median_square_feet_mm', 'median_square_feet_yy', 'total_listing_count',
       'total_listing_count_mm', 'total_listing_count_yy', 'pending_ratio',
       'pending_ratio_mm', 'pending_ratio_yy', 'quality_flag',

### Cross Validation

In [43]:
# Define CV strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# RMSE (negative because sklearn maximizes scores)
rmse_scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1
)

# R² scores
r2_scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="r2",
    cv=kf,
    n_jobs=-1
)

# Convert RMSE back to positive
rmse_scores = -rmse_scores

print("Cross-Validation Results:")
print(f"RMSE: {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f}")
print(f"R²: {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")

Cross-Validation Results:
RMSE: 450765.29 ± 53321.22
R²: nan ± nan


c:\Users\mduro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
1 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\mduro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\mduro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\mduro\AppData\L

# Pytorch

#### Packages

In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV

#### Prepare the data

In [45]:
error_score='raise'

class TorchRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=None, lr=0.0003, epochs=50, batch_size=1024):
        self.input_dim = input_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _build_model(self):
        return nn.Sequential(
            nn.Linear(self.input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(self.device)

    def fit(self, X, y):
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32).reshape(-1, 1)

        self.input_dim = X.shape[1]
        self.model = self._build_model()

        X_tensor = torch.tensor(X).to(self.device)
        y_tensor = torch.tensor(y).to(self.device)

        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        for epoch in range(self.epochs):
            self.model.train()
            perm = torch.randperm(X_tensor.size(0))

            for i in range(0, X_tensor.size(0), self.batch_size):
                idx = perm[i:i+self.batch_size]
                batch_X = X_tensor[idx]
                batch_y = y_tensor[idx]

                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)

                # NaN guard
                if torch.isnan(loss):
                    raise ValueError("Loss became NaN")

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()

        return self

    def predict(self, X):
        self.model.eval()
        X = np.array(X, dtype=np.float32)
        X_tensor = torch.tensor(X).to(self.device)

        with torch.no_grad():
            preds = self.model(X_tensor).cpu().numpy()

        return preds.flatten()

#### Pipeline

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("variance", VarianceThreshold(threshold=0.01)),
    ("scaler", StandardScaler()),
    ("model", TorchRegressor(
        lr=0.0003,
        epochs=50,
        batch_size=1024
        #epochs=30, input_dim=37 # Grid-searched values
    ))
])

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

y = np.log1p(y)

#### Move to GPU (if available)

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Handle both pandas DataFrames and torch Tensors (in case cell is re-run)
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.cpu().numpy()
    elif hasattr(x, 'values'):
        return x.values.astype(np.float32)
    return np.array(x, dtype=np.float32)

# Clean data → numpy (for sklearn pipeline)
X_train_np = np.nan_to_num(to_numpy(X_train), nan=0.0, posinf=0.0, neginf=0.0)
X_test_np = np.nan_to_num(to_numpy(X_test), nan=0.0, posinf=0.0, neginf=0.0)
y_train_np = np.nan_to_num(np.log1p(to_numpy(y_train).ravel()), nan=0.0, posinf=0.0, neginf=0.0)
y_test_np = np.nan_to_num(np.log1p(to_numpy(y_test).ravel()), nan=0.0, posinf=0.0, neginf=0.0)

# GPU tensors (for standalone PyTorch model)
X_train_t = torch.tensor(X_train_np).to(device)
y_train_t = torch.tensor(y_train_np).unsqueeze(1).to(device)
X_test_t = torch.tensor(X_test_np).to(device)
y_test_t = torch.tensor(y_test_np).unsqueeze(1).to(device)

#### Grid Search

#### Training and evaluation

In [49]:
pipeline.fit(X_train_np, y_train_np)
y_pred = pipeline.predict(X_test_np)

# Inverse log1p transform to get original scale
y_test_orig = np.expm1(y_test_np)
y_pred_orig = np.expm1(y_pred)

rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
r2 = r2_score(y_test_orig, y_pred_orig)

print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

RMSE: 11199485.00
R²:   -254.6335


#### Testing

In [51]:
# Check for NaNs / inf
print(np.isnan(X_train_np).sum())
print(np.isinf(X_train_np).sum())

0
0
